1. objective

to analyse customer text data and classify sentiments as Positive, negative, or neutral to support data-driven business decisions.


2. Data understanding and collection

In [4]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns


In [5]:
data = pd.read_csv('/content/drive/MyDrive/ABA/datasets/Sentiment data analysis.csv', encoding='latin-1')
display(data.head())

,textID,text,sentiment,Time of Tweet,Age of User,Country,Population -2020,Land Area (Km²),Density (P/Km²)
0,f87dea47db,Last session of the day http://twitpic.com/67ezh,neutral,morning,0-20,Afghanistan,38928346.0,652860.0,60.0
1,96d74cb729,Shanghai is also really exciting (precisely -...,positive,noon,21-30,Albania,2877797.0,27400.0,105.0
2,eee518ae67,"Recession hit Veronique Branquinho, she has to...",negative,night,31-45,Algeria,43851044.0,2381740.0,18.0
3,01082688c6,happy bday!,positive,morning,46-60,Andorra,77265.0,470.0,164.0
4,33987a8ee5,http://twitpic.com/4w75p - I like it!!,positive,noon,60-70,Angola,32866272.0,1246700.0,26.0


In [6]:
print(data.info(),
data.describe())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4815 entries, 0 to 4814
Data columns (total 9 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   textID            3534 non-null   object 
 1   text              3534 non-null   object 
 2   sentiment         3534 non-null   object 
 3   Time of Tweet     3534 non-null   object 
 4   Age of User       3534 non-null   object 
 5   Country           3534 non-null   object 
 6   Population -2020  3534 non-null   float64
 7   Land Area (Km²)   3534 non-null   float64
 8   Density (P/Km²)   3534 non-null   float64
dtypes: float64(3), object(6)
memory usage: 338.7+ KB
None        Population -2020  Land Area (Km²)  Density (P/Km²)
count      3.534000e+03     3.534000e+03      3534.000000
mean       3.941891e+07     6.722499e+05       348.894171
std        1.468757e+08     1.839134e+06      1967.012367
min        8.010000e+02     0.000000e+00         2.000000
25%        1.968001e+06     

In [7]:
#missing values
print(data.isnull().sum())


textID              1281
text                1281
sentiment           1281
Time of Tweet       1281
Age of User         1281
Country             1281
Population -2020    1281
Land Area (Km²)     1281
Density (P/Km²)     1281
dtype: int64


In [8]:
import re
import nltk
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer
nltk.download('stopwords')
stop_words = set(stopwords.words('english'))
ps = PorterStemmer()
def clean_text(text):
  if isinstance(text, float):
    return ""
  text = text.lower()
  # Remove non-alphabetic characters and replace with spaces
  text = re.sub(r'[^a-z]', ' ', text)
  # Remove extra spaces
  text = re.sub(r'\s+', ' ', text).strip()
  words = text.split()
  words = [ps.stem(word) for word in words if word not in stop_words]
  return ' '.join(words)

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


In [9]:
data['clean_text'] = data['text'].fillna('').apply(clean_text)

In [10]:
x = data['clean_text']
y = data['sentiment']

5. develop model


In [11]:
from sklearn.feature_extraction.text import TfidfVectorizer
vectorizer = TfidfVectorizer(max_features=5000)
x_tfidf = vectorizer.fit_transform(x)


In [12]:
from sklearn.model_selection import train_test_split
x_train, x_test, y_train, y_test = train_test_split(x_tfidf, y, test_size=0.2, random_state=42)

In [ ]:
from sklearn.linear_model import LogisticRegression
nan_indices = y_train.isnull()
x_train_filtered = x_train[~nan_indices.values]
y_train_filtered = y_train[~nan_indices]
model = LogisticRegression()
model.fit(x_train_filtered, y_train_filtered)

6. Analysis

In [ ]:
y_pred = model.predict(x_test)

In [ ]:
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
valid_indices = ~y_test.isnull()
y_test_filtered = y_test[valid_indices]
y_pred_filtered = y_pred[valid_indices]
print("Accuracy:", accuracy_score(y_test_filtered, y_pred_filtered))
print("\nClassification Report:\n", classification_report(y_test_filtered, y_pred_filtered))

In [ ]:
#confusion matrix
conf_matrix = confusion_matrix(y_test_filtered, y_pred_filtered)
sns.heatmap(conf_matrix, annot=True, fmt='d', cmap='rocket')
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.title('Confusion Matrix')
plt.show()

In [ ]:
from wordcloud import WordCloud, STOPWORDS
text_data = ' '.join(data['text'].astype(str))
custom_stopwords = set(STOPWORDS)
custom_stopwords.update(['product','service','company'])

In [ ]:
wordcloud = WordCloud(
    width=800,
    height=400,
    background_color='white',
    stopwords=custom_stopwords,
    max_words=200,
    max_font_size=200,
    random_state=42
)